In [1]:
# Load libraries
import riptide
import cobra
import pandas as pd

In [ ]:
# Load model
jb197 = cobra.io.read_sbml_model("models/355277.4.JB197.sbml")

Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled


In [ ]:
models = [jb197]
values = []
for model in models:
    opt = model.slim_optimize()
    values.append([len(model.reactions), len(model.metabolites), len(model.genes), opt, len(model.medium)])
index = ['JB197']
columns = ['Reactions', 'Metabolites', 'Genes', 'Obj. val.', '# comp. medium']
sum = pd.DataFrame(data = values, index = index, columns = columns)
sum

,Reactions,Metabolites,Genes,Obj. val.,# comp. medium
JB197,1113,1266,952,488.693267,41
HB203,1119,1273,968,490.563199,41


In [4]:
# Load RNA-seq data
rnaSeqjb = pd.read_csv("../../RNA-seq/JB197.csv")
# Load old-new refseq ID dictionary
refseqjb = pd.read_csv('../../IDchanger/locus_tag_uniq_JB197.csv')
# Load KEGGprot file
keggjb = pd.read_csv('models/355277.4.JB197.KEGGprot.out', sep='\t', header = None)

In [ ]:
# refseq df has whitespaces in column and locus ID strings
## Remove whitespace from column name
refseqjb = refseqjb.rename(columns={' old_locus_tag':'old_locus_tag'})
## Remove whitespace from locus IDs
refseqjb['old_locus_tag'] = refseqjb['old_locus_tag'].str.strip()

In [ ]:
# Kegg df: 
## split patric IDs and locus IDs in column 0 --> Save locus IDs to new column
keggjb['old_locus_tag'] = keggjb[0].str.split('|', expand=True)[2]
## Rename column w/Kegg IDs
keggjb = keggjb.rename(columns={1:'keggID'})

In [ ]:
# Merge Kegg IDs with refseq locus tag IDs
id_jb = keggjb[['keggID','old_locus_tag']].merge(refseqjb.dropna(), 
                                               how='left', on='old_locus_tag')

In [ ]:
# Drop rows without a locus tag ID
idjb1 = id_jb[~id_jb['locus_tag'].isna()]

In [ ]:
# Get the genes from model
jbGenes = {g.id for g in jb197.genes} #952 genes (KEGG ids)

In [ ]:
print(len(jbGenes))

952
968


In [ ]:
# Filter ID df to only keep genes that are present in the model
idjb1 = idjb1.loc[idjb1['keggID'].isin(jbGenes)] 
# Could be a good idea to save this dataframe for future reference

In [ ]:
# Merge Kegg gene IDs in the model with rnaSeq data
rnajb1 = idjb1.merge(rnaSeqjb, how='left', left_on='locus_tag', right_on='GeneID')

In [14]:
rnajb1.head()

,keggID,old_locus_tag,locus_tag,JB197_Annotation,GeneID,baseMean,adj_pvalue,log2FoldChange,lfcSE,JB197_29A,...,JB197_29D,JB197_37A,JB197_37B,JB197_37C,JB197_37D,countSUM,DE_pvalue,FCmin,CountMin,DE_Gene
0,lbj:LBJ_0002,LBJ_0002,LBJ_RS00010,branched-chain amino acid transaminase,LBJ_RS00010,2367.522671,5.923008e-02,-0.218643,0.106571,2548.726389,...,2841.672795,1926.744779,2284.485211,2265.586589,2277.648482,18940.18137,False,False,NaN,False
1,metc:MTCT_1105,LBJ_0010,LBJ_RS00050,tRNA dihydrouridine synthase DusB,LBJ_RS00050,4015.940409,1.719710e-14,-0.645956,0.081196,4853.342011,...,5186.742577,3095.917889,3249.641882,3168.032330,3013.036155,32127.52327,True,False,NaN,False
2,cbi:CLJ_B0032,LBJ_0014,LBJ_RS00075,sigma-54-dependent Fis family transcriptional ...,LBJ_RS00075,1564.309012,1.155900e-06,-0.489340,0.095952,1759.959880,...,1757.817013,1240.429851,1353.059194,1203.835063,1407.655205,12514.47209,True,False,NaN,False
3,lbj:LBJ_0018,LBJ_0018,LBJ_RS00095,NaN,LBJ_RS00095,1275.929527,6.733940e-01,-0.099327,0.201003,1236.302914,...,1402.115252,848.517774,1060.215787,1221.057310,1797.783309,10207.43622,False,False,NaN,False
4,ceu:A7L45_06245,LBJ_0030,LBJ_RS00175,ParA family protein,LBJ_RS00175,3569.354374,6.153900e-09,0.607618,0.100090,3005.449758,...,2820.981003,4921.403089,4162.669361,4085.116981,4071.629373,28554.83499,True,False,NaN,False


In [ ]:
# Separate rnaseq data into condition-specific files
rnajb1.iloc[:,[0,9,10,11,12]].to_csv('jb197_29.tsv',sep='\t', index=False) #29 degree
rnajb1.iloc[:,[0,13,14,15,16]].to_csv('jb197_37.tsv',sep='\t', index=False) #37 degree

## Run RIPTiDe

In [ ]:
# Create transcriptomic dictionaries
jb197_29 = riptide.read_transcription_file('jb197_29.tsv', norm = False, header = True) 
jb197_37 = riptide.read_transcription_file('jb197_37.tsv', norm = False, header = True)

In [21]:
# Create Riptide objects
rip_jb29 = riptide.contextualize(model=jb197, conservative=False, transcriptome=jb197_29)
rip_jb37 = riptide.contextualize(model=jb197, conservative=False, transcriptome=jb197_37)


Initializing model and integrating transcriptomic data...
Pruning zero flux subnetworks...
Analyzing context-specific flux distributions...

Reactions pruned to 284 from 1113 (74.48% change)
Metabolites pruned to 291 from 1266 (77.01% change)
Flux through the objective DECREASED to ~445.6775 from ~488.6933 (8.8% change)
Context-specific metabolism correlates with transcriptome (r=0.18, p=0.002 *)

RIPTiDe completed in 21 seconds


Initializing model and integrating transcriptomic data...
Pruning zero flux subnetworks...
Analyzing context-specific flux distributions...

Reactions pruned to 281 from 1113 (74.75% change)
Metabolites pruned to 290 from 1266 (77.09% change)
Flux through the objective DECREASED to ~378.9209 from ~488.6933 (22.46% change)
Context-specific metabolism correlates with transcriptome (r=0.184, p=0.002 *)

RIPTiDe completed in 21 seconds



In [ ]:
riptide.save_output(rip_jb29, path = "./jb197_29", file_type='SBML')
riptide.save_output(rip_jb37, path = "./jb197_37", file_type='SBML')

Saving results to ./jb197_29_20240227_130901
Saving results to ./jb197_29_20240227_130901
Saving results to ./jb197_37_20240227_130922
Saving results to ./jb197_37_20240227_130922
Saving results to ./hb203_29_20240227_130947
Saving results to ./hb203_29_20240227_130947
Saving results to ./hb203_37_20240227_130953
Saving results to ./hb203_37_20240227_130953


In [ ]:
# Load model
jb29 = cobra.io.read_sbml_model("jb197_29_20240227_130901/model.sbml")
jb37 = cobra.io.read_sbml_model("jb197_37_20240227_130922/model.sbml")

In [ ]:
models = [jb197, jb29, jb37]
values = []
for model in models:
    opt = model.slim_optimize()
    values.append([len(model.reactions), len(model.metabolites), len(model.genes), opt, len(model.medium)])
index = ['JB197', 'JB197 29C°', 'JB197 37C°']
columns = ['Reactions', 'Metabolites', 'Genes', 'Obj. val.', '# comp. medium']
sum = pd.DataFrame(data = values, index = index, columns = columns)
sum

,Reactions,Metabolites,Genes,Obj. val.,# comp. medium
JB197,1113,1266,952,488.693267,41
HB203,1119,1273,968,490.563199,41
JB197 29C°,284,291,226,445.677534,34
JB197 37C°,281,290,223,378.920893,33
HB203 29C°,249,260,214,0.000000,35
HB203 37C°,252,264,216,0.000000,35
